# 05 - Retrieval: Finding the Right Context

Retrieval is the **bottleneck** of a RAG system. If you retrieve the wrong chunks, even the best LLM will produce a bad answer. Garbage in, garbage out.

This notebook compares three retrieval strategies:
1. **Similarity search** — straightforward top-k nearest neighbors
2. **MMR (Maximal Marginal Relevance)** — balances relevance with diversity
3. **Similarity with score threshold** — filters by minimum similarity

In [1]:
import os
import sys
import tempfile

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

from dotenv import load_dotenv

load_dotenv(os.path.join(os.path.dirname(os.getcwd()), ".env"))

from rag_engine.chunking.strategies import chunk_documents
from rag_engine.loaders import load_documents
from rag_engine.retrieval.retriever import RetrieverFactory
from rag_engine.vectorstore.chroma_store import add_documents

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data", "sample")

## Setup: Build the Vector Store

In [2]:
# Load, chunk, and embed
md_docs = load_documents(os.path.join(DATA_DIR, "rag_survey.md"))
csv_docs = load_documents(os.path.join(DATA_DIR, "ml_glossary.csv"))
all_docs = md_docs + csv_docs

chunks = chunk_documents(all_docs, strategy="recursive", chunk_size=256, chunk_overlap=30)
print(f"Total chunks: {len(chunks)}")

temp_dir = tempfile.mkdtemp()
vectorstore = add_documents(chunks, collection_name="nb05", persist_directory=temp_dir)

Total chunks: 51


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Strategy 1: Similarity Search (Top-K)

The simplest approach: embed the query, find the k nearest vectors.

In [3]:
query = "What are the different retrieval strategies in RAG?"

sim_retriever = RetrieverFactory.create_retriever(vectorstore, strategy="similarity", top_k=5)
sim_results = sim_retriever.invoke(query)

print("=== Similarity Search (top-5) ===")
for i, doc in enumerate(sim_results, 1):
    print(f"\n[{i}] {doc.page_content[:150]}...")
    print(f"    Source: {doc.metadata.get('source', 'N/A')}")

=== Similarity Search (top-5) ===

[1] term: RAG
definition: Retrieval-Augmented Generation. A technique that enhances LLM responses by retrieving relevant documents from an external knowle...
    Source: C:\Users\mathi\Documents\Github repos\RAG-knowledge-engine\data\sample\ml_glossary.csv

[2] weights during training, RAG systems retrieve pertinent documents from a knowledge base and include them as context in the prompt....
    Source: C:\Users\mathi\Documents\Github repos\RAG-knowledge-engine\data\sample\rag_survey.md

[3] ## Core Components

A typical RAG pipeline consists of the following stages:

### 1. Document Ingestion...
    Source: C:\Users\mathi\Documents\Github repos\RAG-knowledge-engine\data\sample\rag_survey.md

[4] Retrieval-Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge at i...
    Source: C:\Users\mathi\Documents\Github repos\RAG-knowledge-engine\data\sample\rag_survey.md

[5] RA

## Strategy 2: MMR (Maximal Marginal Relevance)

MMR balances **relevance** (how similar to the query) with **diversity** (how different from already-selected documents). This avoids returning 5 near-identical chunks.

The `lambda_mult` parameter controls the trade-off:
- `lambda_mult=1.0` → pure relevance (same as similarity search)
- `lambda_mult=0.0` → pure diversity
- `lambda_mult=0.5` → balanced (recommended)

In [4]:
mmr_retriever = RetrieverFactory.create_retriever(
    vectorstore, strategy="mmr", top_k=5, lambda_mult=0.5
)
mmr_results = mmr_retriever.invoke(query)

print("=== MMR Search (top-5, lambda=0.5) ===")
for i, doc in enumerate(mmr_results, 1):
    print(f"\n[{i}] {doc.page_content[:150]}...")
    print(f"    Source: {doc.metadata.get('source', 'N/A')}")

=== MMR Search (top-5, lambda=0.5) ===

[1] term: RAG
definition: Retrieval-Augmented Generation. A technique that enhances LLM responses by retrieving relevant documents from an external knowle...
    Source: C:\Users\mathi\Documents\Github repos\RAG-knowledge-engine\data\sample\ml_glossary.csv

[2] ## Core Components

A typical RAG pipeline consists of the following stages:

### 1. Document Ingestion...
    Source: C:\Users\mathi\Documents\Github repos\RAG-knowledge-engine\data\sample\rag_survey.md

[3] but too slow for the initial retrieval pass....
    Source: C:\Users\mathi\Documents\Github repos\RAG-knowledge-engine\data\sample\rag_survey.md

[4] ### 5. Retrieval...
    Source: C:\Users\mathi\Documents\Github repos\RAG-knowledge-engine\data\sample\rag_survey.md

[5] After initial retrieval of a larger candidate set (e.g., top-20), use a cross-encoder model to re-score and re-rank the chunks, keeping only the most ...
    Source: C:\Users\mathi\Documents\Github repos\RAG-knowledge

## Side-by-Side Comparison

Let's compare what each strategy retrieves for the same query.

In [5]:
import pandas as pd

comparison = []
for i in range(min(len(sim_results), len(mmr_results))):
    comparison.append({
        "Rank": i + 1,
        "Similarity": sim_results[i].page_content[:80] + "...",
        "MMR": mmr_results[i].page_content[:80] + "...",
    })

df = pd.DataFrame(comparison)
print(df.to_string(index=False))

 Rank                                                                              Similarity                                                                                     MMR
    1    term: RAG\ndefinition: Retrieval-Augmented Generation. A technique that enhances ...    term: RAG\ndefinition: Retrieval-Augmented Generation. A technique that enhances ...
    2     weights during training, RAG systems retrieve pertinent documents from a knowled... ## Core Components\n\nA typical RAG pipeline consists of the following stages:\n\n##...
    3 ## Core Components\n\nA typical RAG pipeline consists of the following stages:\n\n##...                                         but too slow for the initial retrieval pass....
    4     Retrieval-Augmented Generation (RAG) is a technique that enhances Large Language...                                                                     ### 5. Retrieval...
    5    RAG systems should be evaluated on multiple dimensions:\n- **Faithfulness**: Does

## When to Use Which Strategy

| Strategy | Best for | Trade-off |
|----------|---------|----------|
| **Similarity** | Simple Q&A, when you want the most relevant chunks | May return near-duplicates |
| **MMR** | Complex questions spanning multiple topics | Sacrifices some relevance for diversity |
| **Score threshold** | When you want to avoid low-quality results | May return fewer than k results |

**Key insight:** For most RAG applications, **MMR with lambda_mult=0.5** is a good default. It ensures the LLM gets diverse context covering different aspects of the question.

**Next:** [06 - Building the RAG Chain](./06_rag_chain.ipynb) — wiring retrieval into a complete pipeline.